In [ ]:
import pandas as pd

# =============================================================================
# STEP 1: LOAD ALL CSVs
# =============================================================================

print("Step 1: Loading raw CSV files...")

assessments    = pd.read_csv("assessments.csv")
courses        = pd.read_csv("courses.csv")          
student_info   = pd.read_csv("studentInfo.csv")
student_assess = pd.read_csv("studentAssessment.csv")
student_reg    = pd.read_csv("studentRegistration.csv")
student_vle    = pd.read_csv("studentVle.csv")


# =============================================================================
# STEP 2: CLEAN ASSESSMENTS
# =============================================================================

print("\nStep 2: Cleaning assessments...")

assessments = pd.merge(
    assessments,
    courses[['code_module', 'code_presentation', 'module_presentation_length']],
    on=['code_module', 'code_presentation'],
    how='left'
)

missing_date_mask = assessments['date'].isnull()
assessments.loc[missing_date_mask, 'date'] = assessments.loc[missing_date_mask, 'module_presentation_length']
assessments.drop(columns=['module_presentation_length'], inplace=True)

print(f"  Exam dates filled using course length: {missing_date_mask.sum()} rows updated")


# =============================================================================
# STEP 3: CLEAN STUDENT INFO
# =============================================================================

print("\nStep 3: Cleaning student_info...")

n_missing_imd = student_info['imd_band'].isnull().sum()
student_info['imd_band'] = student_info['imd_band'].fillna('Unknown')

print(f"  imd_band: {n_missing_imd} missing values filled with 'Unknown'")


# =============================================================================
# STEP 4: CLEAN STUDENT ASSESSMENTS
# =============================================================================

print("\nStep 4: Cleaning student_assess...")

n_banked = student_assess['is_banked'].sum()
student_assess = student_assess[student_assess['is_banked'] == 0].copy()
print(f"  Removed {n_banked} banked assessment rows")

n_missing_score = student_assess['score'].isnull().sum()
print(f"  Missing scores remaining (no submission recorded): {n_missing_score} — left as NaN intentionally")


# =============================================================================
# STEP 5: CLEAN STUDENT REGISTRATION
# =============================================================================

print("\nStep 5: Cleaning student_reg...")

n_missing_reg = student_reg['date_registration'].isnull().sum()
student_reg['date_registration'] = student_reg['date_registration'].fillna(
    student_reg['date_registration'].median()
)
print(f"  date_registration: {n_missing_reg} missing values filled with median")

student_reg['unregistered'] = student_reg['date_unregistration'].notnull().astype(int)
print(f"  'unregistered' flag created: {student_reg['unregistered'].sum()} students withdrew")


# =============================================================================
# STEP 6: AGGREGATE VLE INTERACTIONS PER STUDENT
# =============================================================================

print("\nStep 6: Aggregating VLE interactions...")

vle_agg = (
    student_vle
    .groupby(['code_module', 'code_presentation', 'id_student'])['sum_click']
    .sum()
    .reset_index()
    .rename(columns={'sum_click': 'total_clicks'})
)

print(f"  VLE aggregated: {vle_agg.shape[0]} student-module rows")


# =============================================================================
# STEP 7: AGGREGATE ASSESSMENT SCORES PER STUDENT
# =============================================================================

print("\nStep 7: Aggregating assessment scores...")

assess_merged = pd.merge(
    student_assess,
    assessments[['id_assessment', 'code_module', 'code_presentation', 'assessment_type']],
    on='id_assessment',
    how='left'
)

assess_agg = (
    assess_merged
    .groupby(['code_module', 'code_presentation', 'id_student'])['score']
    .mean()
    .reset_index()
    .rename(columns={'score': 'mean_score'})
)

submission_counts = (
    assess_merged
    .dropna(subset=['score'])                         
    .groupby(['code_module', 'code_presentation', 'id_student'])['id_assessment']
    .count()
    .reset_index()
    .rename(columns={'id_assessment': 'num_submissions'})
)

assess_agg = pd.merge(assess_agg, submission_counts, on=['code_module', 'code_presentation', 'id_student'], how='left')

print(f"  Assessment scores aggregated: {assess_agg.shape[0]} student-module rows")


# =============================================================================
# STEP 8: BUILD THE MASTER DATAFRAME
# =============================================================================

print("\nStep 8: Building master DataFrame...")

master_df = student_info.copy()

master_df = pd.merge(
    master_df,
    vle_agg,
    on=['code_module', 'code_presentation', 'id_student'],
    how='left'
)

master_df = pd.merge(
    master_df,
    assess_agg,
    on=['code_module', 'code_presentation', 'id_student'],
    how='left'
)

master_df = pd.merge(
    master_df,
    student_reg[['code_module', 'code_presentation', 'id_student', 'date_registration', 'unregistered']],
    on=['code_module', 'code_presentation', 'id_student'],
    how='left'
)

master_df['total_clicks']    = master_df['total_clicks'].fillna(0)
master_df['mean_score']      = master_df['mean_score'].fillna(0)
master_df['num_submissions'] = master_df['num_submissions'].fillna(0)

print(f"  Master DataFrame shape: {master_df.shape}")
print(f"  Null counts:\n{master_df.isnull().sum()[master_df.isnull().sum() > 0]}")


# =============================================================================
# STEP 9: DUPLICATE CHECK
# =============================================================================

print("\nStep 9: Checking for duplicate rows...")

key_cols = ['code_module', 'code_presentation', 'id_student']
n_dupes = master_df.duplicated(subset=key_cols).sum()

if n_dupes > 0:
    print(f"  WARNING: {n_dupes} duplicate rows found — investigate the merge logic!")
else:
    print(f"  No duplicate rows found. ")


# =============================================================================
# STEP 10: ENCODE THE TARGET VARIABLE
# =============================================================================


print("\nStep 10: Encoding target variable (Multi-class)...")

target_map = {
    'Pass':        0,
    'Distinction': 1,
    'Fail':        2,
    'Withdrawn':   3
}
master_df['target'] = master_df['final_result'].map(target_map)

print(f"  Target distribution:\n{master_df['target'].value_counts()}")


# =============================================================================
# STEP 11: PREPARE FEATURES — DROP REDUNDANT COLUMNS
# =============================================================================

print("\nStep 11: Dropping redundant columns...")

cols_to_drop = ['id_student', 'final_result']
df_features = master_df.drop(columns=cols_to_drop)
print(f"  Dropped: {cols_to_drop}")
print(f"  Remaining columns ({df_features.shape[1]}): {list(df_features.columns)}")


# =============================================================================
# STEP 12: ONE-HOT ENCODE CATEGORICAL COLUMNS (TWO VERSIONS)
# =============================================================================

print("\nStep 12: One-hot encoding categorical columns...")

cat_cols = [
    'code_module',
    'code_presentation',
    'gender',
    'region',
    'highest_education',
    'imd_band',
    'age_band',
    'disability'
]

df_encoded_linear = pd.get_dummies(df_features, columns=cat_cols, drop_first=True)
df_encoded_tree = pd.get_dummies(df_features, columns=cat_cols, drop_first=False)

print(f"  Original feature shape     : {df_features.shape}")
print(f"  Encoded shape (linear/SVM) : {df_encoded_linear.shape}  [use with drop_first=True]")
print(f"  Encoded shape (tree-based) : {df_encoded_tree.shape}  [use with drop_first=False]")


# =============================================================================
# STEP 13: FINAL SANITY CHECK
# =============================================================================

print("\n" + "="*60)
print("PIPELINE COMPLETE — Final Summary (Multi-class)")
print("="*60)
print(f"  master_df shape           : {master_df.shape}")
print(f"  df_encoded_linear shape   : {df_encoded_linear.shape}")
print(f"  df_encoded_tree shape     : {df_encoded_tree.shape}")
print(f"\n  Target balance:")

class_names = {0: 'Pass (0)', 1: 'Distinction (1)', 2: 'Fail (2)', 3: 'Withdrawn (3)'}
print(master_df['target'].value_counts(normalize=True).rename(class_names).round(3))

print(f"\n  Null values in encoded (linear):\n  {df_encoded_linear.isnull().sum()[df_encoded_linear.isnull().sum() > 0]}")
print(f"\n  Null values in encoded (tree):\n  {df_encoded_tree.isnull().sum()[df_encoded_tree.isnull().sum() > 0]}")
print("\nReady for model training. Choose df_encoded_linear or df_encoded_tree based on your model.")

Step 1: Loading raw CSV files...
  assessments     : (206, 6)
  courses         : (22, 3)
  student_info    : (32593, 12)
  student_assess  : (173912, 5)
  student_reg     : (32593, 5)
  student_vle     : (10655280, 6)

Step 2: Cleaning assessments...
  Exam dates filled using course length: 11 rows updated

Step 3: Cleaning student_info...
  imd_band: 1111 missing values filled with 'Unknown'

Step 4: Cleaning student_assess...
  Removed 1909 banked assessment rows
  Missing scores remaining (no submission recorded): 172 — left as NaN intentionally

Step 5: Cleaning student_reg...
  date_registration: 45 missing values filled with median
  'unregistered' flag created: 10072 students withdrew

Step 6: Aggregating VLE interactions...
  VLE aggregated: 29228 student-module rows

Step 7: Aggregating assessment scores...
  Assessment scores aggregated: 25580 student-module rows

Step 8: Building master DataFrame...
  Master DataFrame shape: (32593, 17)
  Null counts:
Series([], dtype: int6

In [4]:
from sklearn.model_selection import train_test_split

# =============================================================================
# STEP 1: DEFINE INPUTS AND TARGET
# =============================================================================
# Choose the encoded dataframe you generated in the previous script.
# (Swap to df_encoded_linear if we are going with the Logistic Regression/SVM route)
final_df = df_encoded_tree.copy()

# Separate the features (X) from the answer key (y)
X = final_df.drop(columns=['target'])
y = final_df['target']

print(f"Total dataset size : {X.shape[0]} rows")


# =============================================================================
# STEP 2: SPLIT OFF THE TEST SET (15%)
# =============================================================================
# stratify=y ensures the 4 classes (Pass, Distinction, Fail, Withdrawn) 
# are distributed equally across the splits.
# random_state=42 locks the randomness so you get the exact same split every time.

X_temp, X_test, y_temp, y_test = train_test_split(
    X, 
    y, 
    test_size=0.15, 
    random_state=42, 
    stratify=y
)


# =============================================================================
# STEP 3: SPLIT THE REMAINDER INTO TRAIN (70%) AND VALIDATION (15%)
# =============================================================================
# We have 85% of the data left in 'temp'. 
# To get 15% of the ORIGINAL total for validation, we calculate: 15 / 85 = 0.1764
val_ratio = 0.15 / 0.85

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, 
    y_temp, 
    test_size=val_ratio, 
    random_state=42, 
    stratify=y_temp
)


# =============================================================================
# STEP 4: VERIFY THE SPLITS
# =============================================================================
print("\n" + "=" * 50)
print("DATA SPLIT COMPLETE")
print("=" * 50)

print(f"Training Set   : {X_train.shape[0]} rows ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation Set : {X_val.shape[0]} rows ({len(X_val)/len(X)*100:.1f}%)")
print(f"Testing Set    : {X_test.shape[0]} rows ({len(X_test)/len(X)*100:.1f}%)")

# Verify that stratification worked by checking the class ratios in the test set
class_names = {0: 'Pass', 1: 'Distinction', 2: 'Fail', 3: 'Withdrawn'}
print("\nTarget Distribution in Test Set:")
print(y_test.value_counts(normalize=True).rename(class_names).round(3))

Total dataset size : 32593 rows

DATA SPLIT COMPLETE
Training Set   : 22815 rows (70.0%)
Validation Set : 4889 rows (15.0%)
Testing Set    : 4889 rows (15.0%)

Target Distribution in Test Set:
target
Pass           0.379
Withdrawn      0.312
Fail           0.216
Distinction    0.093
Name: proportion, dtype: float64
